<a href="https://colab.research.google.com/github/caiopsd/video-classifier/blob/main/genvidbench_full_hpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GenVidBench Pair1 Completo - Treino Temporal com Otimizacao de Hiperparametros

Este notebook treina um classificador temporal usando o dataset completo do Pair1 no Drive.

Protocolo:
- montar dataset completo do `Pair1_labels.txt`
- split estratificado em `train`, `validation` e `test`
- usar **apenas train/validation** para otimizacao de hiperparametros
- usar `test` **somente depois** de escolher os melhores hiperparametros

Convencao de labels: `hf_label = 1` (fake), `hf_label = 0` (real).


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q optuna decord


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 55.4 MB/s eta 0:00:00


In [2]:
import os
import gc
import math
import time
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

try:
    import optuna
    HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False

try:
    from decord import VideoReader, cpu
    HAS_DECORD = True
except Exception:
    HAS_DECORD = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)
print('HAS_OPTUNA:', HAS_OPTUNA)
print('HAS_DECORD:', HAS_DECORD)


DEVICE: cuda
HAS_OPTUNA: True
HAS_DECORD: True


In [ ]:
DATASET_ROOT = Path('/content/drive/MyDrive/GenVidBench/GenVidBench')

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Nao encontrei DATASET_ROOT: {DATASET_ROOT}')

label_files = [DATASET_ROOT / 'Pair1_labels.txt']
label_files = [p for p in label_files if p.exists()]

if not label_files:
    raise FileNotFoundError('Nao encontrei Pair1_labels.txt no DATASET_ROOT')

print('Usando somente label file do Pair1:')
for lf in label_files:
    print('-', lf)


Usando somente label file do Pair1:
- /content/drive/MyDrive/GenVidBench/GenVidBench/Pair1_labels.txt


In [ ]:
# Leitura do dataset completo apenas do Pair1
rows = []
for lf in label_files:
    split_name = lf.stem.replace('_labels', '').lower()
    with lf.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rel_path, label_str = line.rsplit(' ', 1)
                hf_label = int(label_str)
            except Exception:
                continue

            if hf_label not in (0, 1):
                continue

            rel_path = rel_path.replace('\\', '/')
            abs_path = DATASET_ROOT / rel_path
            source_folder = Path(rel_path).parts[1] if len(Path(rel_path).parts) > 1 else 'unknown'

            rows.append({
                'dataset_split_file': split_name,
                'rel_path': rel_path,
                'abs_path': str(abs_path),
                'source_folder': source_folder,
                'hf_label': hf_label,
                'exists': abs_path.exists(),
            })

df = pd.DataFrame(rows)
df = df[df['exists']].reset_index(drop=True)

print('Total de exemplos validos:', len(df))
print('Distribuicao hf_label (0=real,1=fake):')
print(df['hf_label'].value_counts(dropna=False))
print('\nTop pastas origem:')
print(df['source_folder'].value_counts().head(10))
df.head()


Total de exemplos validos: 74135
Distribuicao hf_label (0=real,1=fake):
hf_label
1    54004
0    20131
Name: count, dtype: int64

Top pastas origem:
source_folder
vript    20131
ms       13501
pika     13501
t2vz     13501
vc2      13501
Name: count, dtype: int64


,dataset_split_file,rel_path,abs_path,source_folder,hf_label,exists
0,pair1,Pair1/ms/ms-0015c714-5c20-5766-8944-540102ec09...,/content/drive/MyDrive/GenVidBench/GenVidBench...,ms,1,True
1,pair1,Pair1/ms/ms-001c5b92-f5c3-577d-a80f-cde08a9ef1...,/content/drive/MyDrive/GenVidBench/GenVidBench...,ms,1,True
2,pair1,Pair1/ms/ms-002a629a-e7db-51c0-8613-cb5ab2cbf7...,/content/drive/MyDrive/GenVidBench/GenVidBench...,ms,1,True
3,pair1,Pair1/ms/ms-002a6bed-74b8-547b-bf58-f7abd5fd67...,/content/drive/MyDrive/GenVidBench/GenVidBench...,ms,1,True
4,pair1,Pair1/ms/ms-002acde4-7101-5176-b1e1-fb9ba768b7...,/content/drive/MyDrive/GenVidBench/GenVidBench...,ms,1,True


In [ ]:
# Redução controlada por source_folder
# fake (hf_label=1): ms, pika, t2vz, vc2 -> 2000 cada
# real (hf_label=0): vript -> 5000

fake_sources = ["ms", "pika", "t2vz", "vc2"]
real_source = "vript"

parts = []

# Fakes
for src in fake_sources:
    chunk = df[(df["hf_label"] == 1) & (df["source_folder"] == src)]
    n = min(2000, len(chunk))
    if n == 0:
        print(f"[WARN] Nenhum exemplo para fake source={src}")
        continue
    parts.append(chunk.sample(n=n, random_state=SEED))

# Reais
real_chunk = df[(df["hf_label"] == 0) & (df["source_folder"] == real_source)]
n_real = min(5000, len(real_chunk))
if n_real == 0:
    raise ValueError("Nenhum exemplo real encontrado em source_folder='vript'.")
parts.append(real_chunk.sample(n=n_real, random_state=SEED))

# Dataset final reduzido
df_use = pd.concat(parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Total selecionado:", len(df_use))
print("\nDistribuição por hf_label:")
print(df_use["hf_label"].value_counts())
print("\nDistribuição por source_folder:")
print(df_use["source_folder"].value_counts())

Total selecionado: 13000

Distribuição por hf_label:
hf_label
1    8000
0    5000
Name: count, dtype: int64

Distribuição por source_folder:
source_folder
vript    5000
pika     2000
t2vz     2000
ms       2000
vc2      2000
Name: count, dtype: int64


In [ ]:
# Salvar df_use no Google Drive para reuso rápido
drive_data_dir = Path("/content/drive/MyDrive/GenVidBench/prepared_data")
drive_data_dir.mkdir(parents=True, exist_ok=True)

df_use_parquet = drive_data_dir / "df_use_pair1.parquet"
df_use_csv = drive_data_dir / "df_use_pair1.csv"

df_use.to_parquet(df_use_parquet, index=False)
df_use.to_csv(df_use_csv, index=False, encoding="utf-8")

print("Salvo:")
print("-", df_use_parquet)
print("-", df_use_csv)
print("shape:", df_use.shape)

In [3]:
df_use_path = Path("/content/drive/MyDrive/GenVidBench/prepared_data/df_use_pair1.parquet")
df_use = pd.read_parquet(df_use_path)

print("df_use carregado:", df_use.shape)
print(df_use.head())
print(df_use["hf_label"].value_counts())

df_use carregado: (13000, 6)
  dataset_split_file                                           rel_path  \
0              pair1  Pair1/pika/pika-bc3ee8d4-8597-562e-9ac0-5e4334...   
1              pair1  Pair1/pika/pika-adfe7168-4de3-5ba3-900d-0d5b35...   
2              pair1  Pair1/t2vz/t2vz-78315228-ef87-503e-ad37-44b071...   
3              pair1              Pair1/vript/-AZJlsx67aQ-Scene-018.mp4   
4              pair1  Pair1/ms/ms-acf98c22-bf88-5d61-b80c-10e4f87878...   

                                            abs_path source_folder  hf_label  \
0  /content/drive/MyDrive/GenVidBench/GenVidBench...          pika         1   
1  /content/drive/MyDrive/GenVidBench/GenVidBench...          pika         1   
2  /content/drive/MyDrive/GenVidBench/GenVidBench...          t2vz         1   
3  /content/drive/MyDrive/GenVidBench/GenVidBench...         vript         0   
4  /content/drive/MyDrive/GenVidBench/GenVidBench...            ms         1   

   exists  
0    True  
1    True  
2  

In [4]:
# Split estratificado: train 70%, val 15%, test 15%
train_df, temp_df = train_test_split(
    df_use,
    test_size=0.30,
    stratify=df_use['hf_label'],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['hf_label'],
    random_state=SEED,
)

print(f'Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}')
print('Train label dist:', train_df['hf_label'].value_counts())
print('Val label dist: ', val_df['hf_label'].value_counts())
print('Test label dist: ', test_df['hf_label'].value_counts())


Train=9100 | Val=1950 | Test=1950
Train label dist: hf_label
1    5600
0    3500
Name: count, dtype: int64
Val label dist:  hf_label
1    1200
0     750
Name: count, dtype: int64
Test label dist:  hf_label
1    1200
0     750
Name: count, dtype: int64


In [5]:
class TemporalVideoDataset(Dataset):
    def __init__(self, frame_df, num_frames=12, image_size=128):
        self.df = frame_df.reset_index(drop=True)
        self.num_frames = num_frames
        self.image_size = image_size
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.df)

    def _sample_indices(self, total):
        if total <= 0:
            return np.zeros(self.num_frames, dtype=np.int64)
        if total >= self.num_frames:
            return np.linspace(0, total - 1, self.num_frames).astype(np.int64)
        idx = np.linspace(0, total - 1, total).astype(np.int64).tolist()
        while len(idx) < self.num_frames:
            idx.append(idx[-1])
        return np.array(idx, dtype=np.int64)

    def _read_video(self, path):
        if HAS_DECORD:
            vr = VideoReader(path, ctx=cpu(0))
            idx = self._sample_indices(len(vr))
            return vr.get_batch(idx).asnumpy()

        import cv2
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        idx = set(self._sample_indices(total).tolist())
        frames = []
        i = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if i in idx:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
            i += 1
        cap.release()
        if len(frames) == 0:
            raise RuntimeError('Falha leitura de video')
        while len(frames) < self.num_frames:
            frames.append(frames[-1])
        return np.stack(frames[:self.num_frames], axis=0)

    def _to_tensor_clip(self, frames_np):
        x = torch.from_numpy(frames_np).float() / 255.0
        x = x.permute(0, 3, 1, 2)
        out = []
        for t in range(x.shape[0]):
            fr = T.functional.resize(x[t], [self.image_size, self.image_size], antialias=True)
            fr = self.normalize(fr)
            out.append(fr)
        x = torch.stack(out, dim=0)
        x = x.permute(1, 0, 2, 3)  # C,T,H,W
        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            frames = self._read_video(row['abs_path'])
            clip = self._to_tensor_clip(frames)
        except Exception:
            clip = torch.zeros(3, self.num_frames, self.image_size, self.image_size)

        label = torch.tensor(float(row['hf_label']), dtype=torch.float32)
        return clip, label


In [6]:
class TemporalGRUNet(nn.Module):
    def __init__(self, hidden_size=128, dropout=0.2):
        super().__init__()
        bb = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        self.encoder = bb.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.feature_dim = 576

        self.gru = nn.GRU(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2 + 1, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        b, c, t, h, w = x.shape
        fr = x.permute(0, 2, 1, 3, 4).reshape(b * t, c, h, w)

        feat = self.encoder(fr)
        feat = self.pool(feat).flatten(1)
        feat = feat.reshape(b, t, self.feature_dim)

        if t > 1:
            delta = feat[:, 1:, :] - feat[:, :-1, :]
            motion = delta.abs().mean(dim=(1, 2), keepdim=False).unsqueeze(1)
        else:
            motion = torch.zeros((b, 1), device=x.device)

        out, _ = self.gru(feat)
        temporal_embed = out[:, -1, :]
        fused = torch.cat([temporal_embed, motion], dim=1)
        return self.head(fused).squeeze(1)


In [7]:
def make_loaders(train_df, val_df, cfg):
    tr_ds = TemporalVideoDataset(train_df, num_frames=cfg['num_frames'], image_size=cfg['image_size'])
    va_ds = TemporalVideoDataset(val_df, num_frames=cfg['num_frames'], image_size=cfg['image_size'])

    train_loader = DataLoader(
        tr_ds,
        batch_size=cfg['batch_size'],
        shuffle=True,
        num_workers=cfg['num_workers'],
        pin_memory=(DEVICE == 'cuda'),
    )
    val_loader = DataLoader(
        va_ds,
        batch_size=cfg['batch_size'],
        shuffle=False,
        num_workers=cfg['num_workers'],
        pin_memory=(DEVICE == 'cuda'),
    )
    return train_loader, val_loader

def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    criterion = nn.BCEWithLogitsLoss()

    losses, y_true, y_pred = [], [], []

    for clips, labels in loader:
        clips = clips.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            logits = model(clips)
            loss = criterion(logits, labels)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        probs = torch.sigmoid(logits)
        pred = (probs >= 0.5).long()

        losses.append(loss.item())
        y_true.extend(labels.detach().cpu().long().tolist())
        y_pred.extend(pred.detach().cpu().tolist())

    avg_loss = float(np.mean(losses)) if losses else 0.0
    acc = accuracy_score(y_true, y_pred) if y_true else 0.0
    f1 = f1_score(y_true, y_pred, zero_division=0) if y_true else 0.0
    return avg_loss, acc, f1

def train_one_config(train_df, val_df, cfg):
    train_loader, val_loader = make_loaders(train_df, val_df, cfg)

    model = TemporalGRUNet(hidden_size=cfg['hidden_size'], dropout=cfg['dropout']).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

    best_val_f1 = -1.0
    best_state = None

    for epoch in range(1, cfg['epochs'] + 1):
        tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, optimizer=optimizer)
        va_loss, va_acc, va_f1 = run_epoch(model, val_loader, optimizer=None)

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"epoch={epoch:02d} tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} | va_loss={va_loss:.4f} va_f1={va_f1:.4f}"
        )

    del train_loader, val_loader, model, optimizer
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    return best_val_f1, best_state


In [8]:
# CHECKPOINTS DURANTE HPO (salva cada trial no Drive)

CKPT_DIR = Path("/content/drive/MyDrive/GenVidBench/hpo_checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = CKPT_DIR / "hpo_trials_log.jsonl"
BEST_PATH = CKPT_DIR / "best_so_far.pt"

def save_trial_checkpoint(
    trial_id: int,
    cfg: dict,
    best_val_f1: float,
    best_state: dict,
    extra: dict | None = None,
):
    """
    Salva checkpoint do melhor estado daquele trial.
    """
    ckpt_path = CKPT_DIR / f"trial_{trial_id:03d}_best.pt"

    payload = {
        "trial_id": trial_id,
        "cfg": cfg,
        "best_val_f1": float(best_val_f1),
        "state_dict": best_state,   # já está em CPU no seu train_one_config
        "seed": SEED,
        "extra": extra or {},
    }

    torch.save(payload, ckpt_path)

    # Log append-only para auditoria/recovery
    with open(META_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "trial_id": trial_id,
            "ckpt_path": str(ckpt_path),
            "best_val_f1": float(best_val_f1),
            "cfg": cfg,
        }, ensure_ascii=False) + "\n")

    return ckpt_path

def update_global_best_if_needed(
    trial_id: int,
    cfg: dict,
    best_val_f1: float,
    best_state: dict,
):
    """
    Mantém um 'best_so_far.pt' único para retomar rápido.
    """
    current_best = -1.0
    if BEST_PATH.exists():
        try:
            old = torch.load(BEST_PATH, map_location="cpu")
            current_best = float(old.get("best_val_f1", -1.0))
        except Exception:
            current_best = -1.0

    if best_val_f1 > current_best:
        torch.save({
            "trial_id": trial_id,
            "cfg": cfg,
            "best_val_f1": float(best_val_f1),
            "state_dict": best_state,
            "seed": SEED,
        }, BEST_PATH)
        print(f"[HPO] Novo best global salvo: trial={trial_id} f1={best_val_f1:.4f}")


In [ ]:
# Espaco de busca
BASE_CFG = {
    'num_workers': 2,
    'epochs': 3,  # para HPO rapido; aumente depois
}

N_TRIALS = 9

def sample_random_cfg(rng):
    return {
        'num_frames': int(rng.choice([8, 12, 16])),
        'image_size': int(rng.choice([112, 128, 160])),
        'batch_size': int(rng.choice([2, 4, 6]) if DEVICE == 'cuda' else rng.choice([1, 2])),
        'hidden_size': int(rng.choice([96, 128, 192, 256])),
        'dropout': float(rng.choice([0.1, 0.2, 0.3, 0.4])),
        'lr': float(rng.choice([1e-4, 2e-4, 3e-4, 5e-4])),
        'weight_decay': float(rng.choice([1e-5, 1e-4, 3e-4])),
    }

best_cfg = None
best_f1 = -1.0
best_state = None
history = []

if HAS_OPTUNA:
    def objective(trial):
        cfg = {
            'num_frames': trial.suggest_categorical('num_frames', [8, 12, 16]),
            'image_size': trial.suggest_categorical('image_size', [112, 128, 160]),
            'batch_size': trial.suggest_categorical('batch_size', [2, 4, 6] if DEVICE == 'cuda' else [1, 2]),
            'hidden_size': trial.suggest_categorical('hidden_size', [96, 128, 192, 256]),
            'dropout': trial.suggest_categorical('dropout', [0.1, 0.2, 0.3, 0.4]),
            'lr': trial.suggest_categorical('lr', [1e-4, 2e-4, 3e-4, 5e-4]),
            'weight_decay': trial.suggest_categorical('weight_decay', [1e-5, 1e-4, 3e-4]),
        }
        cfg.update(BASE_CFG)
        print('\nTrial', trial.number, cfg)
        val_f1, state = train_one_config(train_df, val_df, cfg)

        # salva checkpoint do trial
        trial_ckpt = save_trial_checkpoint(
            trial_id=trial.number,
            cfg=cfg,
            best_val_f1=val_f1,
            best_state=state,
        )
        return val_f1

    study = optuna.create_study(direction='maximize', study_name='genvidbench_full_hpo')
    study.optimize(objective, n_trials=N_TRIALS)

    best_cfg = study.best_trial.params
    best_cfg.update(BASE_CFG)
    best_f1, best_state = train_one_config(train_df, val_df, best_cfg)
    print('\nBest trial params:', best_cfg)
    print('Best val F1:', best_f1)
else:
    rng = np.random.default_rng(SEED)
    for t in range(N_TRIALS):
        cfg = sample_random_cfg(rng)
        cfg.update(BASE_CFG)
        print('\nRandom trial', t, cfg)

        val_f1, state = train_one_config(train_df, val_df, cfg)
        history.append({'trial': t, 'val_f1': val_f1, **cfg})

        # salva checkpoint do trial
        trial_ckpt = save_trial_checkpoint(
            trial_id=t,
            cfg=cfg,
            best_val_f1=val_f1,
            best_state=state,
        )
        print(f"[HPO] Trial checkpoint salvo em: {trial_ckpt}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_cfg = cfg
            best_state = state

    print('\nBest random cfg:', best_cfg)
    print('Best val F1:', best_f1)


[I 2026-05-05 20:49:22,947] A new study created in memory with name: genvidbench_full_hpo



Trial 0 {'num_frames': 12, 'image_size': 128, 'batch_size': 4, 'hidden_size': 192, 'dropout': 0.3, 'lr': 0.0001, 'weight_decay': 0.0001, 'num_workers': 2, 'epochs': 3}
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 180MB/s]


epoch=01 tr_loss=0.4235 tr_f1=0.8441 | va_loss=0.3061 va_f1=0.8901
epoch=02 tr_loss=0.2830 tr_f1=0.9069 | va_loss=0.2040 va_f1=0.9357
epoch=03 tr_loss=0.2149 tr_f1=0.9344 | va_loss=0.2080 va_f1=0.9385


[I 2026-05-05 23:24:43,157] Trial 0 finished with value: 0.9385281385281385 and parameters: {'num_frames': 12, 'image_size': 128, 'batch_size': 4, 'hidden_size': 192, 'dropout': 0.3, 'lr': 0.0001, 'weight_decay': 0.0001}. Best is trial 0 with value: 0.9385281385281385.


...


Trial 6 {'num_frames': 16, 'image_size': 160, 'batch_size': 4, 'hidden_size': 256, 'dropout': 0.1, 'lr': 0.0002, 'weight_decay': 1e-05, 'num_workers': 2, 'epochs': 3}
epoch=01 tr_loss=0.3654 tr_f1=0.8727 | va_loss=0.1760 va_f1=0.9482
epoch=02 tr_loss=0.2386 tr_f1=0.9246 | va_loss=0.1604 va_f1=0.9541
epoch=03 tr_loss=0.1757 tr_f1=0.9454 | va_loss=0.1402 va_f1=0.9575


[I 2026-05-06 08:12:59,522] Trial 6 finished with value: 0.957464553794829 and parameters: {'num_frames': 16, 'image_size': 160, 'batch_size': 4, 'hidden_size': 256, 'dropout': 0.1, 'lr': 0.0002, 'weight_decay': 1e-05}. Best is trial 6 with value: 0.957464553794829.



Trial 7 {'num_frames': 16, 'image_size': 128, 'batch_size': 4, 'hidden_size': 96, 'dropout': 0.2, 'lr': 0.0005, 'weight_decay': 0.0003, 'num_workers': 2, 'epochs': 3}
epoch=01 tr_loss=0.4233 tr_f1=0.8484 | va_loss=0.2771 va_f1=0.9124
epoch=02 tr_loss=0.3062 tr_f1=0.8992 | va_loss=0.3212 va_f1=0.8971
epoch=03 tr_loss=0.2447 tr_f1=0.9211 | va_loss=0.1670 va_f1=0.9554


[I 2026-05-06 09:51:42,316] Trial 7 finished with value: 0.9553827261563651 and parameters: {'num_frames': 16, 'image_size': 128, 'batch_size': 4, 'hidden_size': 96, 'dropout': 0.2, 'lr': 0.0005, 'weight_decay': 0.0003}. Best is trial 6 with value: 0.957464553794829.



Trial 8 {'num_frames': 8, 'image_size': 112, 'batch_size': 6, 'hidden_size': 96, 'dropout': 0.1, 'lr': 0.0005, 'weight_decay': 1e-05, 'num_workers': 2, 'epochs': 3}
epoch=01 tr_loss=0.3569 tr_f1=0.8746 | va_loss=0.1888 va_f1=0.9436


In [ ]:
# Treino final com melhores hiperparametros usando train+val
trainval_df = pd.concat([train_df, val_df]).reset_index(drop=True)

final_cfg = dict(best_cfg)
final_cfg['epochs'] = max(5, best_cfg['epochs'] + 2)

trainval_loader = DataLoader(
    TemporalVideoDataset(trainval_df, num_frames=final_cfg['num_frames'], image_size=final_cfg['image_size']),
    batch_size=final_cfg['batch_size'],
    shuffle=True,
    num_workers=final_cfg['num_workers'],
    pin_memory=(DEVICE == 'cuda'),
)

test_loader = DataLoader(
    TemporalVideoDataset(test_df, num_frames=final_cfg['num_frames'], image_size=final_cfg['image_size']),
    batch_size=final_cfg['batch_size'],
    shuffle=False,
    num_workers=final_cfg['num_workers'],
    pin_memory=(DEVICE == 'cuda'),
)

final_model = TemporalGRUNet(hidden_size=final_cfg['hidden_size'], dropout=final_cfg['dropout']).to(DEVICE)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=final_cfg['lr'], weight_decay=final_cfg['weight_decay'])

for epoch in range(1, final_cfg['epochs'] + 1):
    tr_loss, tr_acc, tr_f1 = run_epoch(final_model, trainval_loader, optimizer=optimizer)
    print(f'[FINAL] epoch={epoch:02d} loss={tr_loss:.4f} acc={tr_acc:.4f} f1={tr_f1:.4f}')

torch.save({
    'state_dict': final_model.state_dict(),
    'cfg': final_cfg,
    'seed': SEED,
}, 'genvidbench_full_best_model.pt')
print('Modelo salvo em genvidbench_full_best_model.pt')


In [ ]:
# Salvar no Google Drive
drive_ckpt_dir = Path("/content/drive/MyDrive/GenVidBench/checkpoints")
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

ckpt_path = drive_ckpt_dir / "genvidbench_pair1_best_model.pt"

torch.save(
    {
        "state_dict": final_model.state_dict(),
        "cfg": final_cfg,
        "seed": SEED,
    },
    ckpt_path,
)

print(f"Modelo salvo em: {ckpt_path}")

In [ ]:
# AVALIACAO FINAL NO TESTE (somente apos HPO + treino final)
final_model.eval()
all_true, all_pred = [], []

with torch.no_grad():
    for clips, labels in test_loader:
        clips = clips.to(DEVICE)
        logits = final_model(clips)
        probs = torch.sigmoid(logits)
        pred = (probs >= 0.5).long().cpu().numpy()

        all_pred.extend(pred.tolist())
        all_true.extend(labels.long().numpy().tolist())

print('Test accuracy:', accuracy_score(all_true, all_pred))
print('Test F1:', f1_score(all_true, all_pred, zero_division=0))
print('\nClassification report (0=real,1=fake):')
print(classification_report(all_true, all_pred, target_names=['real', 'fake'], zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(all_true, all_pred))


## Observacoes praticas

- Este notebook usa apenas o Pair1 completo.
- `MAX_PER_CLASS = None` usa todo o Pair1 (mais demorado).
- Para acelerar HPO, mantenha `epochs` baixo durante a busca e aumente no treino final.
- Se faltar memoria GPU, reduza `batch_size`, `image_size` ou `num_frames`.
